In [46]:
from functools import partial

import pandas as pd
import jax
import jax.tree_util as jtu
import jax.numpy as jnp

from functools import partial

import equinox as eqx
from equinox.nn import MLP, Identity, Linear

from jaxtyping import PyTree, Array
from typing import Callable

import matplotlib.pyplot as plt


In [2]:
jax.config.update("jax_disable_jit", False)
jax.config.update("jax_debug_nans", True)
jax.config.update("jax_debug_infs", True)
jax.config.update("jax_enable_x64", True)
from rich.traceback import install
install()

# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"
# InteractiveShell.showtraceback = "verbose"

# import sys
# def full_traceback_exception_hook(exctype, value, traceback):
#     import traceback as tb
#     tb.print_exception(exctype, value, traceback)

# sys.excepthook = full_traceback_exception_hook

<bound method InteractiveShell.excepthook of <ipykernel.zmqshell.ZMQInteractiveShell object at 0x7f9fbccfdd90>>

# Parameter and coefficients

In [67]:
# Number of time step
nt = 5

# Time duration
t_length = 2.

# Time step
dt = t_length / nt

# The maximum age step
τ_max = 3

# The old-water concentration
C_old = jnp.array([0.5]) # [mg/l]

# The first-order reaction rate constant
k1 = jnp.array([0.0])

# The equilibrium concentration
C_eq = jnp.array([0.0]) # [mg/o]

# Fractionation
α_Q, α_ET = 1., 0.


# Flow and solutes in/out

In [48]:
pulse_start1 = 0.05
pulse_end1 = 0.45

pulse_start2 = 1.25
pulse_end2 = 1.45

C_tracer_input1 = 0.5
C_tracer_input2 = 1.5
Q_steady = 1.
Storage_vol = 0.1

data_df = pd.DataFrame(index=jnp.arange(nt) * dt)
data_df['Q'] = Q_steady
data_df['ET'] = Q_steady
data_df['J'] = Q_steady
data_df['C_J'] = 0
data_df.loc[pulse_start1:pulse_end1, 'C_J'] = C_tracer_input1
data_df.loc[pulse_start1:pulse_end1, 'J'] = Q_steady * 2
data_df.loc[pulse_start1:pulse_end1, 'Q'] = Q_steady / 20.
data_df.loc[pulse_start2:pulse_end2, 'Q'] = Q_steady / 50.


/tmp/ipykernel_81776/1264661931.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data_df.loc[pulse_start1:pulse_end1, 'C_J'] = C_tracer_input1


# SAS functions

In [49]:
# The base class
class SASBase(eqx.Module):
    loc: Array
    scale: Array
    
    def __init__(self, loc=0.0, scale=1.0):
        self.loc = jnp.array(loc)
        self.scale = jnp.array(scale)
    
    def pdf(self, Si, x):
        # Return PDF
        raise Exception('Not implemented')
    
    def __call__(self, Si, x):
        # Return CDF:
        raise Exception('Not implemented')


In [50]:
# The base class
class SAS_null(SASBase):
    loc: Array
    scale: Array
    
    def pdf(self, Si, x):
        # Return PDF
        return 0.
    
    def __call__(self, Si, x):
        # Return CDF:
        return 0.


In [51]:
# Gamme distribution
class SAS_Gamma(SASBase):
    a: Array
    # func: Callable

    # TODO: the minimum scaled y is needed to avoid
    # the FloatingPointError in jit operations!
    
    def __init__(self, a, loc=0.0, scale=1.0):
        super().__init__(loc, scale)
        self.a = jnp.array(a)
        # self.func = MLP(in_size=1, out_size=2, key=jax.random.key(42), **mlp_kwargs)
    
    def pdf(self, Si, x=None):
        a, loc, scale = self.a, self.loc, self.scale
        y = (Si - loc) / scale
        y = jax.lax.max(1e-20, y)
        return jax.scipy.stats.gamma.pdf(y, a, loc=0., scale=1.)
    
    def __call__(self, Si, x=None):
        a, loc, scale = self.a, self.loc, self.scale
        y = (Si - loc) / scale
        y = jax.lax.max(1e-20, y)
        return jax.scipy.stats.gamma.cdf(y, a, loc=0., scale=1.)


In [52]:
# Beta distribution
class SAS_Beta(SASBase):
    a: Array
    b: Array
    
    def __init__(self, a, b, loc=0.0, scale=1.0):
        super().__init__(loc, scale)
        self.a, self.b = jnp.array(a), jnp.array(b)
    
    def pdf(self, Si, x=None):
        a, b, loc, scale = self.a, self.b, self.loc, self.scale
        return jax.scipy.stats.beta.pdf(Si, a, b, loc, scale)
    
    def __call__(self, Si, x=None):
        a, b, loc, scale = self.a, self.b, self.loc, self.scale
        return jax.scipy.stats.beta.cdf(Si, a, b, loc, scale)


In [53]:
# Kumaraswamy distribution 
class SAS_Kumaraswamy(SASBase):
    a: Array
    b: Array
    
    def __init__(self, a, b, loc=0.0, scale=1.0):
        super().__init__(loc, scale)
        self.a, self.b = jnp.array(a), jnp.array(b)
    
    def pdf(self, Si, x=None):
        a, b, loc, scale = self.a, self.b, self.loc, self.scale
        y = (Si - loc) / scale
        y = jax.lax.min(jax.lax.max(0., y), 1.)
        return a * b * y ** (a-1) * (1-y**a) ** (b-1)
    
    def __call__(self, Si, x=None):
        # Return CDF:
        a, b, loc, scale = self.a, self.b, self.loc, self.scale
        y = (Si - loc) / scale
        y = jax.lax.min(jax.lax.max(0., y), 1.)
        return 1. - (1. - y**a) ** b


In [54]:
# Mixture density network
class SAS_MDN(SASBase):
    m: eqx.Module
    m_α: eqx.Module
    m_μ: eqx.Module
    m_σ: eqx.Module
    # pdf_func: Callable
    # cdf_func: Callable
    
    def __init__(self, n_input, n_hidden, n_mixture, key, loc=0., scale=1., **mlp_kwargs):
        super().__init__(loc, scale)
        key1, key2, key3, key4 = jax.random.split(key,4)
        
        # MLP model for predicting the hidden states
        # n_mlp_output = (n_output+2) * n_mixture
        self.m = MLP(in_size=n_input, out_size=n_hidden, key=key1, **mlp_kwargs)
        
        # MLP model for predicting α, μ, and σ
        self.m_α = Linear(in_features=n_hidden, out_features=n_mixture, key=key2)
        self.m_μ = Linear(in_features=n_hidden, out_features=n_mixture, key=key3)
        self.m_σ = Linear(in_features=n_hidden, out_features=n_mixture, key=key4)
        
        # # PDF function of mixture distributions 
        # self.pdf_func = jax.scipy.stats.norm.pdf
        
        # # CDF function of mixture distributions 
        # self.cdf_func = jax.scipy.stats.norm.cdf
    
    def get_param(self, x):
        # Calculate the weights, mean, and standard deviation of the mixture distributions
        z = self.m(x)  # shape: (n_hidden,)
        z_α, z_μ, z_σ = self.m_α(z), self.m_μ(z), self.m_σ(z) # shape: (n_mixture,) (n_mixture,) (n_mixture)
        
        # The weights (n_mixture,)
        α = jax.nn.softmax(z_α)
        
        # The mean (n_mixture,)
        μ = z_μ
        
        # The standard deviation (n_mixture,)
        σ = jnp.exp(z_σ)
        
        return α, μ, σ
    
    def pdf(self, Si, x):
        y = (Si - self.loc) / self.scale
        α, μ, σ = self.get_param(x)
        pdfs = jax.vmap(jax.scipy.stats.norm.pdf, in_axes=(None,0,0))(y, μ, σ)
        return jnp.sum(jnp.dot(α, pdfs))
    
    def __call__(self, Si, x):
        y = (Si - self.loc) / self.scale
        α, μ, σ = self.get_param(x)
        cdfs = jax.vmap(jax.scipy.stats.norm.cdf, in_axes=(None,0,0))(y, μ, σ)
        return jnp.sum(jnp.dot(α, cdfs))


# SAS-based transport forward function

In [55]:
# Compute mQ -- mass loss with flow
def compute_mQ(sT, mT, α, Q_t, pQ):
    # sT, mT, α, Q_t, pQ: (1,), (1,), (1,), (1,), (1,)
    # return mT / sT * (α * Q_t * pQ)
    # return jnp.where(sT == 0.0, 0.0, mT / sT * (α * Q_t * pQ))
    part_a = mT * (α * Q_t * pQ)
    y = jnp.where(sT == 0.0, 1.0, sT)
    y = jnp.where(sT == 0.0, 0.0, 1./ y)
    return part_a * y

# Compute mR -- Reaction rate
def compute_mR(sT, mT, k1, C_eq):
    # sT, mT, k1, C_eq: (1,), (1,), (1,), (1,)
    return k1 * (C_eq*sT - mT)

compute_mR_vec = jax.vmap(compute_mR, in_axes=(None,0,0,0))


In [56]:
def f(sTmT, ST, dt, 
      J_τ_t, C_J_τ_t, Q_t, ET_t, 
      sas_Q, sas_ET, sas_arg_Q, sas_arg_ET, 
      α_Q, α_ET, k1, C_eq
):
    # sTmT: (1+nm,)
    # ST: (1,)
    # dt: (1,)
    # J_τ_t: (1,)
    # C_J_τ_t: (nm,)
    # Q_t: (1,)
    # ET_t: (1,)
    # α_Q: (1,)
    # α_ET: (1,)
    # sas_Q: (1,)
    # sas_arg_Q: (1,)
    # sas_ET: (1,)
    # sas_arg_ET: (1,)
    # k1: (nm,)
    # C_eq: (nm,)
    
    sT = sTmT[0] # (1,)
    mT = sTmT[1:] # (nm,)
    
    # Calculate the pQ, mQ, and mR
    PQ1, PQ2 = sas_Q(ST+sT*dt, sas_arg_Q), sas_Q(ST, sas_arg_Q)
    pQ = (PQ1 - PQ2) / dt
    PET1, PET2 = sas_ET(ST+sT*dt, sas_arg_ET), sas_ET(ST, sas_arg_ET)
    pET = (PET1 - PET2) / dt
    mQ = compute_mQ(sT, mT, α_Q, Q_t, pQ) # (nm,)
    mET = compute_mQ(sT, mT, α_ET, ET_t, pET) # (nm,)

    pQET = jnp.array([pQ, pET])  # (2,)
    mQET = jnp.array([mQ, mET]).T  # (nm, 2)
    
    # Calculate mR
    mR = compute_mR_vec(sT, mT, k1, C_eq)  # (nm,)
    
    # Update the change of sT and mT
    δs = (J_τ_t - Q_t * pQ * dt - ET_t * pET * dt) / dt  # (1,)

    # Update the change of sT and mT
    δm = (J_τ_t * C_J_τ_t - mQ * dt - mET * dt + mR * dt) / dt  # (nm,)

    # print(pQET.shape)
    return jnp.concat([jnp.array([δs]), δm]), pQET, mQET, mR


fvec = jax.vmap(
    f, in_axes=(0,0,None,0,0,0,0,None,None,0,0,
                None,None,None,None)
)


# SAS transport solver/functions

In [57]:
# !!!! Note that the usage of nn.relu would affect the accuracy of implicit differentiation
def solve_step_euler(states, ST_top, ST_bot, dt, *args, f):
    # states: (nt, ns)
    # ST_top, ST_bot: (nt)

    # Note: we use relu to make sure the updated states are non-zero
    ST = ST_top
    r, pQET, mQET, mR = f(states, ST, dt, *args)

    # states_new = states + r * dt
    states_new = jax.nn.relu(states + r * dt)

    return states_new, pQET, mQET, mR


In [58]:
# !!!! Note that the usage of nn.relu would affect the accuracy of implicit differentiation
def solve_step_rk2(states, ST_top, ST_bot, dt, *args, f):
    # states: (nt, ns)
    # ST_top, ST_bot: (nt)

    # Note: we use relu to make sure the updated states are non-zero
    ST1 = ST_top
    states1 = states
    r1, pQET1, mQET1, mR1 = f(states1, ST1, dt, *args)
    
    ST2 = ST_bot
    states2 = jax.nn.relu(states+1.*r1*dt)
    # states2 = states+1.*r1*dt
    r2, pQET2, mQET2, mR2 = f(states2, ST2, dt, *args)

    pQET = 1./2 * (pQET1 + pQET2)
    mQET = 1./2 * (mQET1 + mQET2)
    mR = 1./2 * (mR1 + mR2)
    states_new = jax.nn.relu(states + (r1 + r2)/2. * dt)
    # states_new = states + (r1 + r2)/2. * dt

    return states_new, pQET, mQET, mR


In [59]:
# !!!! Note that the usage of nn.relu would affect the accuracy of implicit differentiation
def solve_step_rk4(states, ST_top, ST_bot, dt, *args, f):
    # states: (nt, ns)
    # ST_top, ST_bot: (nt)

    # Note: we use relu to make sure the updated states are non-zero
    ST1 = ST_top
    states1 = states
    r1, pQET1, mQET1, mR1 = f(states1, ST1, dt, *args)
    # jax.debug.print('r1: {x}', x=r1)
    
    ST2 = ST_top/2 + ST_bot/2
    states2 = jax.nn.relu(states+0.5*r1*dt)
    # states2 = states+0.5*r1*dt
    r2, pQET2, mQET2, mR2 = f(states2, ST2, dt, *args)
    # jax.debug.print('r2: {x}', x=r2)
    
    ST3 = ST_top/2 + ST_bot/2
    states3 = jax.nn.relu(states+0.5*r2*dt)
    # states3 = states+0.5*r2*dt
    r3, pQET3, mQET3, mR3 = f(states3, ST3, dt, *args)
    # jax.debug.print('r3: {x}', x=r3)
    
    ST4 = ST_bot
    states4 = jax.nn.relu(states+1.*r3*dt)
    # states4 = states+1.*r3*dt
    r4, pQET4, mQET4, mR4 = f(states4, ST4, dt, *args)
    # jax.debug.print('r4: {x}', x=r4)

    # states_new = states + 1./6 * (r1 + 2 * r2 + 2 * r3 + r4) * dt
    states_new = jax.nn.relu(states + 1./6 * (r1 + 2 * r2 + 2 * r3 + r4) * dt)
    pQET = 1./6 * (pQET1 + 2 * pQET2 + 2 * pQET3 + pQET4)
    mQET = 1./6 * (mQET1 + 2 * mQET2 + 2 * mQET3 + mQET4)
    mR = 1./6 * (mR1 + 2 * mR2 + 2 * mR3 + mR4)

    # jax.debug.print('states_new: {x}', x=states_new)

    return states_new, pQET, mQET, mR


In [60]:
def solve_ttd_sTmT(
    f, solver,
    J_full, C_J, Q, ET, dt,
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
    sas_Q, sas_ET,
    α_Q, α_ET, k1, C_eq, 
    # *, 
    # solver, f
):
    # J_full: (nτ, nt)
    # C_J: (nt,)
    # Q: (nt,)
    # ET: (nt,)
    # dt: (1,)
    # sTmT_0: (nt, 1+nm)
    # ST_top_0: (nt,)
    # ST_bot_0: (nt,)
    # sTmT_init: (nτ, 1+nm)
    # sas_Q, sas_ET: (1,)
    # α_Q, α_ET: (1,)
    # k1, C_eq: (nm,)
    
    sas_funcs = [sas_Q, sas_ET]
    other_args = [α_Q, α_ET, k1, C_eq]
    
    def step_τ(states, x):
        # sTmT_init_τ : (1+nm,)
        sTmT_init_τ, J_τ = x
        sT_init_τ = sTmT_init_τ[0]
    
        # sTmT : (nt, 1+nm)
        # ST_top, ST_bot : (nt+1,)
        sTmT, ST_top, ST_bot = states
        
        # Get fluxes and sas arguments
        fluxes = [J_τ, C_J, Q, ET]
        sas_args = [Q[:,None], ET[:,None]]  # TODO: this should be the function input!
    
        # Solve the ODE system
        # sTmT_new: (nt, 1+nm)
        # pQET_new: (nt, 2)
        # mQET_new: (nt, nm, 2)
        # mRET_new: (nt, nm)
        sTmT_new, pQET_new, mQET_new, mRET_new = solver(
            sTmT, ST_top[:-1], ST_bot[1:], dt, 
            *fluxes, *sas_funcs, *sas_args, *other_args,
            f=f
        )
        sT_new = sTmT_new[...,0]

        # Update ST
        ST_top_new = ST_bot
        ST_bot_new = jnp.concat([sT_init_τ[None] * dt, ST_bot[1:] + sT_new * dt])
        # jax.debug.print('sT_new: {x}; ST_bot: {y}; args: {z}', x=sT_new, y=ST_bot, z=args)
        
        # Variables as the initial condition to the next step
        sTmT_new_rotate = jnp.concat([sTmT_init_τ[None,...], sTmT_new[:-1]], axis=0)
        
        # Variables to save
        state_to_save = [sTmT_new, mQET_new, pQET_new]
        # jax.debug.print("sTmT_new: {x}", x=sTmT_new)
        
        return (sTmT_new_rotate, ST_top_new, ST_bot_new), state_to_save
    
    _, state_to_save = jax.lax.scan(step_τ, (sTmT_0, ST_top_0, ST_bot_0), (sTmT_init,J_full))

    sTmTs, mQETs, pQETs = state_to_save
    # jax.debug.print("sTmTs: {x}", x=sTmTs)

    return sTmTs, mQETs, pQETs


In [61]:
def solve_sas(
    solve_ttd_func, solver, f,
    J_full, C_J, Q, ET, dt,
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
    sas_Q, sas_ET,
    α_Q, α_ET, k1, C_eq, C_Q_old, P_Q_old,
    # *, 
    # solve_ttd_func, solver, f
):
    # J_full: (nτ, nt)
    # C_J: (nt, nm)
    # Q: (nt,)
    # ET: (nt,)
    # dt: (1,)
    # sTmT_0: (nt, 1+nm)
    # ST_top_0: (nt,)
    # ST_bot_0: (nt,)
    # sTmT_init: (nτ, 1+nm)
    # sas_Q, sas_ET: (1,)
    # α_Q, α_ET: (1,) 
    # k1, C_eq: (nm,)
    # C_Q_0, P_old: (nm,), (nt,)
    
    nτ, nt = J_full.shape

    # Solve the TTDs' ODE and return
    # sTmTs: (nτ, nt, nm+1)
    # mQETs: (nτ, nt, nm, 2)
    # pQETs: (nτ, nt, 2)
    sTmTs, mQETs, pQETs = solve_ttd_func(
        f, solver,
        J_full, C_J, Q, ET, dt,
        sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
        sas_Q, sas_ET, α_Q, α_ET, k1, C_eq, 
        # solver=solver, f=f
    )
    
    sT = sTmTs[...,0]  # (nτ, nt)
    mT = sTmTs[...,1:]  # (nτ, nt, nm)
    
    sT_init, mT_init = sTmT_init[:,0], sTmT_init[:,1:]
    sT = jnp.concat([sT_init[:,None], sT], axis=1)  # (nτ, nt+1)
    mT = jnp.concat([mT_init[:,None,:], mT], axis=1)  # (nτ, nt+1, nm)

    # Calculate C_Q
    mQs = mQETs[...,0]  # (nτ, nt, nm)
    C_Q1 = jax.vmap(lambda a,b: a/b, in_axes=(0,0))(
        mQs.sum(axis=0) * dt, Q
    )  # (nt, nm)

    # Contribution from old water
    pQs = pQETs[...,0]  # (nτ, nt)
    P_Q_old = P_Q_old - pQs.sum(axis=0) * dt  # (nt,)
    # C_Q2 = jax.vmap(lambda a,b: a*b, in_axes=(0,0))(
    #     C_Q_old[None,:], P_Q_old[:,None]
    # )  # (nt,nm)
    C_Q2 = jnp.vectorize(lambda a,b: a*b)(
        jnp.stack([C_Q_old]*nt), jnp.stack([P_Q_old]*nm).T
    )  # (nt,nm)

    C_Q = C_Q1 + C_Q2  # (nt, nm)

    # jax.debug.print("C_Q: {x}", x=C_Q)
    
    return sT, mT, mQETs, pQETs, C_Q


# Run SAS

In [62]:
C_J = jnp.array(data_df[['C_J']].values) # (nt,nm)
Q = jnp.array(data_df['Q'].values) # (nt,)
ET = jnp.array(data_df['ET'].values) # (nt,)
J = jnp.array(data_df['J'].values) # (nt,)
J_full = jnp.concat([J[None,:], jnp.zeros([τ_max-1, nt])])

# Initial sT and mT at time = 0
# sT_init, mT_init = jnp.zeros(τ_max), jnp.zeros(τ_max)
sT_init, mT_init = jnp.array([10., 15., 20.]), jnp.array([[10., 10., 10.]]).T
sTmT_init = jnp.concat([sT_init[...,None], mT_init], axis=1)
nm = mT_init.shape[1]

# Initials at age = 0
sTmT_0 = jnp.zeros([nt, 1+nm])
ST_top_0, ST_bot_0 = jnp.zeros(nt+1), jnp.zeros(nt+1)
P_Q_old = jnp.ones(nt) 
# C_Q_0, P_Q_old = jnp.zeros([nt, nm]), jnp.ones(nt) 
# initials = [sT_0, mT_0, ST_top_0, ST_bot_0, C_Q_0, P_old]

# SAS function
# sas_func = partial(sas_gamma, a=0.02)
# sas_func = partial(sas_beta, a=2.31, b=0.627)


## Kumaraswamy

In [65]:
sas_Q = SAS_Kumaraswamy(a=2.31, b=0.627, loc=0., scale=1.)
sas_ET = SAS_Kumaraswamy(a=2.31, b=0.627, loc=0., scale=1.)
# sas_ET = SAS_null(loc=0., scale=1.)


In [68]:
sT_final, mT_final, mQETs_final, pQETs_final, C_Q_final = solve_sas(
    solve_ttd_sTmT, solve_step_rk4, fvec,
    J_full, C_J, Q, ET, dt,
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
    sas_Q, sas_ET,
    α_Q, α_ET, k1, C_eq, C_old, P_Q_old,
    # solve_ttd_func=solve_ttd_sTmT, solver=solve_step_rk4, f=fvec
)
# sT_final, mT_final, mQETs_final, pQETs_final, C_Q_final
C_Q_final


Array([[1.03198326],
       [0.76578462],
       [0.37709572],
       [0.52958264],
       [0.33207822]], dtype=float64)

## Gamma

In [37]:
# eqx.filter_jvp(sas_Q, (jnp.array(1e-40),), (jnp.array(0.),))
# # sas_Q(-0.1)


In [42]:
sas_Q = SAS_Gamma(a=0.5)
sas_ET = SAS_null()


In [43]:
sT_final, mT_final, mQETs_final, pQETs_final, C_Q_final = solve_sas(
    solve_ttd_sTmT, solve_step_rk4, fvec,
    J_full, C_J, Q, ET, dt,
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
    sas_Q, sas_ET,
    α_Q, α_ET, k1, C_eq, C_old, P_Q_old,
    # solve_ttd_func=solve_ttd_sTmT, solver=solve_step_rk4, f=fvec
)
# sT_final, mT_final, mQETs_final, pQETs_final, C_Q_final
C_Q_final


(5, 1)
(1,) (5,)
(5, 1)


Array([[0.65350304],
       [0.5459742 ],
       [0.22572112],
       [0.11662002],
       [0.        ]], dtype=float64)

## MDN

In [44]:
n_input, n_hidden, n_mixture, key = 1, 20, 5, jax.random.key(42)
mlp_kwargs = {"width_size": 20, "depth":3}
sas_Q = SAS_MDN(n_input, n_hidden, n_mixture, key, **mlp_kwargs)
sas_ET = SAS_null()


In [45]:
sT_final, mT_final, mQETs_final, pQETs_final, C_Q_final = solve_sas(
    solve_ttd_sTmT, solve_step_rk4, fvec,
    J_full, C_J, Q, ET, dt,
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
    sas_Q, sas_ET,
    α_Q, α_ET, k1, C_eq, C_old, P_Q_old,
    # solve_ttd_func=solve_ttd_sTmT, solver=solve_step_rk4, f=fvec
)
# sT_final, mT_final, mQETs_final, pQETs_final, 
C_Q_final


(5, 1)
(1,) (5,)
(5, 1)


Array([[0.45047285],
       [0.32403894],
       [0.12569741],
       [0.09208174],
       [0.        ]], dtype=float64)

# Customize the derivative using IFT

## Without IFT

In [30]:
def evolve(sas_Q):
    sT_final, mT_final, mQETs_final, pQETs_final, C_Q_final = solve_sas(
            solve_ttd_sTmT, solve_step_rk4, fvec,
            J_full, C_J, Q, ET, dt,
            sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
            sas_Q, sas_ET,
            α_Q, α_ET, k1, C_eq, C_Q_0, P_Q_old,
            # solve_ttd_func=solve_ttd_sTmT, solver=solve_step_rk4, f=fvec
    )
    # return C_Q_final.sum()
    return sT_final[-1,-2]


In [109]:
# grad = eqx.filter_grad(evolve)(sas_Q, α_Q, α_ET, k1, C_eq, C_Q_0, P_Q_old)
# grad = jax.grad(evolve, argnums=0)(sas_Q, α_Q, α_ET, k1, C_eq, C_Q_0, P_Q_old)
grad = eqx.filter_grad(evolve)(sas_Q)
# grad.loc, grad.ab
grad


SAS_MDN(
  loc=f64[],
  scale=f64[],
  m=MLP(
    layers=(
      Linear(
        weight=f64[20,1],
        bias=f64[20],
        in_features=1,
        out_features=20,
        use_bias=True
      ),
      Linear(
        weight=f64[20,20],
        bias=f64[20],
        in_features=20,
        out_features=20,
        use_bias=True
      ),
      Linear(
        weight=f64[20,20],
        bias=f64[20],
        in_features=20,
        out_features=20,
        use_bias=True
      ),
      Linear(
        weight=f64[20,20],
        bias=f64[20],
        in_features=20,
        out_features=20,
        use_bias=True
      )
    ),
    activation=None,
    final_activation=None,
    use_bias=True,
    use_final_bias=True,
    in_size=1,
    out_size=20,
    width_size=20,
    depth=3
  ),
  m_α=Linear(
    weight=f64[5,20],
    bias=f64[5],
    in_features=20,
    out_features=5,
    use_bias=True
  ),
  m_μ=Linear(
    weight=f64[5,20],
    bias=f64[5],
    in_features=20,
    out_features

In [112]:
# eqx.filter_jvp(evolve, (sas_Q,), (sas_Q,))

In [30]:
%timeit eqx.filter_grad(evolve)(k1)

3.34 s ± 56.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## (TODO) With IFT

In [87]:
def solve_ttd_sTmT_aug(
    f, solver,
    aug_J_full, aug_C_J, aug_Q, aug_ET, dt,
    aug_sTmT_0, aug_ST_top_0, aug_ST_bot_0, aug_sTmT_init,  
    aug_sas_Q, aug_sas_ET,
    aug_α_Q, aug_α_ET, aug_k1, aug_C_eq, 
    # *, 
    # solver, f
):
    # aug_J_full: (nτ, nt, 2)
    # aug_C_J: (nt, 2)
    # aug_Q: (nt, 2)
    # aug_ET: (nt, 2)
    # dt: (1,)
    # aug_sTmT_0: (nt, 1+nm, 2)
    # aug_ST_top_0: (nt+1, 2)
    # aug_ST_bot_0: (nt+1, 2)
    # aug_sTmT_init: (nτ, 1+nm, 2)
    # aug_sas_Q, aug_sas_ET: (2,)
    # aug_α_Q, aug_α_ET: (2,)
    # aug_k1, aug_C_eq: (nm, 2)

    aug_sas_funcs = [aug_sas_Q, aug_sas_ET]
    aug_other_args = [aug_α_Q, aug_α_ET, aug_k1, aug_C_eq]
    
    def step_τ(states, x):
        # aug_sTmT_init_τ : (1+nm, 2)
        # aug_J_τ: (nt, 2)
        aug_sTmT_init_τ, aug_J_τ = x
        aug_sT_init_τ = aug_sTmT_init_τ[0,:]  # (2,)
    
        # aug_sTmT : (nt, 1+nm, 2)
        # aug_ST_top, ST_bot : (nt+1, 2)
        aug_sTmT, aug_ST_top, aug_ST_bot = states
        
        # Get fluxes and sas arguments
        aug_fluxes = [aug_J_τ, aug_C_J, aug_Q, aug_ET]

        # TODO: this should be the function input!
        aug_sas_args_Q = [aug_Q[:,[0]], aug_Q[:,[1]]]
        aug_sas_args_ET = [aug_ET[:,[0]], aug_ET[:,[1]]]
        aug_sas_args = [aug_sas_args_Q, aug_sas_args_ET]
    
        # Solve the ODE system
        # aug_sTmT_new: (nt, 1+nm, 2)
        # aug_pQET_new: (nt, 2, 2)
        # aug_mQET_new: (nt, nm, 2, 2)
        # aug_mRET_new: (nt, nm, 2)
        aug_sTmT_new, aug_pQET_new, aug_mQET_new, aug_mRET_new = solver(
            aug_sTmT, aug_ST_top[:-1,:], aug_ST_bot[1:,:], dt, 
            *aug_fluxes, *aug_sas_funcs, *aug_sas_args, *aug_other_args,
            f=f
        )
        aug_sT_new = aug_sTmT_new[:,0,:]  # (nt, 2)
        # jax.debug.print('aug_sTmT_new: {x}', x=aug_sTmT_new[...,:])

        # Update ST
        aug_ST_top_new = aug_ST_bot  # (nt+1, 2)
        aug_ST_bot_new = jnp.concat(
            [aug_sT_init_τ[None,:] * dt, aug_ST_bot[1:,:] + aug_sT_new * dt], axis=0
        )  # (nt+1, 2)
        
        # Variables as the initial condition to the next step
        aug_sTmT_new_rotate = jnp.concat(
            [aug_sTmT_init_τ[None,...], aug_sTmT_new[:-1,:,:]], axis=0
        )  # (nt, 1+nm, 2)

        # Variables to save
        state_to_save = [aug_sTmT_new, aug_mQET_new, aug_pQET_new]
        
        return (aug_sTmT_new_rotate, aug_ST_top_new, aug_ST_bot_new), state_to_save
    
    _, state_to_save = jax.lax.scan(
        step_τ, (aug_sTmT_0, aug_ST_top_0, aug_ST_bot_0), (aug_sTmT_init, aug_J_full)
    )

    aug_sTmTs, aug_mQETs, aug_pQETs = state_to_save
    # jax.debug.print("sTmTs: {x}", x=aug_sTmTs[...,0])
    # jax.debug.print("tan_sTmTs: {x}", x=aug_sTmTs[...,1])

    return aug_sTmTs, aug_mQETs, aug_pQETs


In [88]:
solve_ttd_sTmT_ift = jax.custom_jvp(solve_ttd_sTmT, nondiff_argnums=(0,1))
# solve_ttd_sTmT_ift = eqx.filter_custom_jvp(solve_ttd_sTmT)

@solve_ttd_sTmT_ift.defjvp
def solve_ttd_sTmT_ift_jvp(f, solver, primals, tangents):
    J_full, C_J, Q, ET, dt,\
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init,\
    sas_Q, sas_ET,\
    α_Q, α_ET, k1, C_eq = primals
    
    δJ_full, δC_J, δQ, δET, _,\
    δsTmT_0, δST_top_0, δST_bot_0, δsTmT_init,\
    δsas_Q, δsas_ET,\
    δα_Q, δα_ET, δk1, δC_eq = tangents

    # TODO: 
    def f_aug(
        aug_sTmT, aug_ST, dt, 
        aug_J_τ, aug_C_J_τ,
        aug_Q, aug_ET, 
        aug_sas_Q, aug_sas_ET,
        aug_sas_arg_Q, aug_sas_arg_ET,
        aug_α_Q, aug_α_ET, aug_k1, aug_C_eq
    ):
        primals = [
            aug_sTmT[...,0], aug_ST[...,0], dt, 
            aug_J_τ[...,0], aug_C_J_τ[...,0],
            aug_Q[...,0], aug_ET[...,0], 
            aug_sas_Q[0], aug_sas_ET[0],
            aug_sas_arg_Q[0], aug_sas_arg_ET[0],
            aug_α_Q[0], aug_α_ET[0], aug_k1[...,0], aug_C_eq[...,0]
        ]
        tangents = [
            aug_sTmT[...,1], aug_ST[...,1], dt, 
            aug_J_τ[...,1], aug_C_J_τ[...,1],
            aug_Q[...,1], aug_ET[...,1],
            aug_sas_Q[1], aug_sas_ET[1],
            aug_sas_arg_Q[0], aug_sas_arg_ET[0],  # TODO!!
            aug_α_Q[1], aug_α_ET[1], aug_k1[...,1], aug_C_eq[...,1]
        ]
        # primal_dot, tangent_dot = eqx.filter_jvp(f, primals, tangents)
        primal_dot, tangent_dot = jax.jvp(f, primals, tangents)
        states_dot, pQET_dot, mQET_dot, mR_dot = primal_dot
        δstates_dot, δpQET_dot, δmQET_dot, δmR_dot = tangent_dot
        # jax.debug.print('aug_Q: {x}',  x=aug_Q[...,1])
        # jax.debug.print('δstates_dot: {x}',  x=δstates_dot)
        # jax.debug.print('states_dot_all: {x}',  x=jnp.stack([states_dot, δstates_dot], axis=-1))
        return jnp.stack([states_dot, δstates_dot], axis=-1), jnp.stack([pQET_dot, δpQET_dot], axis=-1), \
               jnp.stack([mQET_dot, δmQET_dot], axis=-1), jnp.stack([mR_dot, δmR_dot], axis=-1)
    
    aug_states = solve_ttd_sTmT_aug(
        f_aug, solver,
        jnp.stack([J_full, δJ_full], axis=-1), jnp.stack([C_J, δC_J], axis=-1), 
        jnp.stack([Q, δQ], axis=-1), jnp.stack([ET, δET], axis=-1), dt,
        jnp.stack([sTmT_0, δsTmT_0], axis=-1), jnp.stack([ST_top_0, δST_top_0], axis=-1),
        jnp.stack([ST_bot_0, ST_bot_0], axis=-1), jnp.stack([sTmT_init, δsTmT_init], axis=-1),
        [sas_Q, δsas_Q], [sas_ET, δsas_ET],
        jnp.stack([α_Q, δα_Q], axis=-1), jnp.stack([α_ET, δα_ET], axis=-1),
        jnp.stack([k1, δk1], axis=-1), jnp.stack([C_eq, δC_eq], axis=-1),
    )
    aug_sTmTs, aug_mQETs, aug_pQETs = aug_states

    # TODO: Check the shape
    sTmTs, sTmTs_dot = aug_sTmTs[...,0], aug_sTmTs[...,1]
    mQETs, mQETs_dot = aug_mQETs[...,0], aug_mQETs[...,1]
    pQETs, pQETs_dot = aug_pQETs[...,0], aug_pQETs[...,1]
    # print(aug_pQETs)
    return (sTmTs, mQETs, pQETs), (sTmTs_dot, mQETs_dot, pQETs_dot)


In [82]:
def evolve_ift(Q):
    sT_final, mT_final, mQETs_final, pQETs_final, C_Q_final = solve_sas(
            solve_ttd_sTmT_ift, solve_step_rk4, fvec,
            J_full, C_J, Q, ET, dt,
            sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
            sas_Q, sas_ET,
            α_Q, α_ET, k1, C_eq, C_Q_0, P_Q_old,
            # solve_ttd_func=solve_ttd_sTmT_ift, solver=solve_step_rk4, f=fvec
    )
    # return C_Q_final.sum()
    return sT_final[-1,-2]


In [90]:
grad = eqx.filter_grad(evolve_ift)(Q)
grad


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 grad = eqx.filter_grad(evolve_ift)(Q)                                                        │
│   2 grad                                                                                         │
│   3                                                                                              │
│                                                                                                  │
│ /global/cfs/cdirs/m1800/peishi/.conda/envs/watershed/lib/python3.12/site-packages/equinox/_ad.py │
│ :96 in __call__                                                                                  │
│                                                                                                  │
│     93 │   │   return self._fun_value_and_grad.__wrapped__  # pyright: ignore                    │
│     94 │                                                                                         │
│     95 │   def __call__(self, /, *args, **kwargs):                                               │
│ ❱   96 │   │   value, grad = self._fun_value_and_grad(*args, **kwargs)                           │
│     97 │   │   if self._has_aux:                                                                 │
│     98 │   │   │   _, aux = value                                                                │
│     99 │   │   │   return grad, aux                                                              │
│                                                                                                  │
│ /global/cfs/cdirs/m1800/peishi/.conda/envs/watershed/lib/python3.12/site-packages/equinox/_ad.py │
│ :79 in __call__                                                                                  │
│                                                                                                  │
│     76 │   │   │   │   )                                                                         │
│     77 │   │   x, *args = args                                                                   │
│     78 │   │   diff_x, nondiff_x = partition(x, is_inexact_array)                                │
│ ❱   79 │   │   return fun_value_and_grad(diff_x, nondiff_x, *args, **kwargs)                     │
│     80 │                                                                                         │
│     81 │   def __get__(self, instance, owner):                                                   │
│     82 │   │   if instance is None:                                                              │
│                                                                                                  │
│ /global/cfs/cdirs/m1800/peishi/.conda/envs/watershed/lib/python3.12/site-packages/jax/_src/trace │
│ back_util.py:180 in reraise_with_filtered_traceback                                              │
│                                                                                                  │
│   177   def reraise_with_filtered_traceback(*args, **kwargs):                                    │
│   178 │   __tracebackhide__ = True                                                               │
│   179 │   try:                                                                                   │
│ ❱ 180 │     return fun(*args, **kwargs)                                                          │
│   181 │   except Exception as e:                                                                 │
│   182 │     mode = _filtering_mode()                                                             │
│   183 │     if _is_under_reraiser(e) or mode == "off":                                           │
│                                                            

In [89]:
eqx.filter_jvp(evolve_ift, (Q,), (Q+0.2,))

r4: [[[ 1.18042778 -3.31765714]
  [ 0.          0.        ]]

 [[ 4.90118538 -5.42373282]
  [ 2.45059269 -2.71186641]]

 [[ 1.18042778 -3.31765714]
  [ 0.          0.        ]]

 [[ 1.18042778 -3.31765714]
  [ 0.          0.        ]]

 [[ 1.18042778 -3.31765714]
  [ 0.          0.        ]]]
states_new: [[[0.63642933 0.        ]
  [0.         0.        ]]

 [[1.97251193 0.        ]
  [0.98625596 0.        ]]

 [[0.63642933 0.        ]
  [0.         0.        ]]

 [[0.63642933 0.        ]
  [0.         0.        ]]

 [[0.63642933 0.        ]
  [0.         0.        ]]]
r4: [[[-1.17722006 -0.2833735 ]
  [-1.17722006 -0.2833735 ]]

 [[-0.00750495 -0.03617305]
  [ 0.          0.        ]]

 [[-0.724899   -0.53498197]
  [-0.3624495  -0.26749098]]

 [[-0.29681728 -0.30006717]
  [ 0.          0.        ]]

 [[-0.29681728 -0.30006717]
  [ 0.          0.        ]]]
states_new: [[[9.3496199  0.        ]
  [9.3496199  0.        ]]

 [[0.6276026  0.        ]
  [0.         0.        ]]

 [[1.52106

(Array(1.28782965, dtype=float64), Array(0., dtype=float64))

In [446]:
%timeit eqx.filter_grad(evolve_ift)(k1)

3.58 s ± 72.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


# Compute gradients and Jacobian

In [34]:
def evolve_C_Q(sas_Q):
    sT_final, mT_final, mQETs_final, pQETs_final, C_Q_final = solve_sas(
            solve_ttd_sTmT, solve_step_rk4, fvec,
            J_full, C_J, Q, ET, dt,
            sTmT_0, ST_top_0, ST_bot_0, sTmT_init,  
            sas_Q, sas_ET,
            α_Q, α_ET, k1, C_eq, C_Q_0, P_Q_old,
            # solve_ttd_func=solve_ttd_sTmT, solver=solve_step_rk4, f=fvec
    )
    # return C_Q_final.sum()
    return C_Q_final


In [28]:
# eqx.filter_grad(evolve)(sas_Q)

In [29]:
# jax.jacfwd(evolve_C_Q)(sas_Q)

In [35]:
eqx.filter_jacfwd(evolve_C_Q)(sas_Q)

SAS_Gamma(loc=f64[5,1], scale=f64[5,1], a=f64[5,1])

In [25]:
# def f1(a, b):
#     return jnp.ones(5) + a - 4 * b

# eqx.filter_jacfwd(f1)(jnp.array(4.), jnp.array(3.))

In [24]:
# class A(eqx.Module):
#     a: Array
#     b: Array

#     def __init__(self, a, b):
#         self.a = jnp.array(a)
#         self.b = jnp.array(b)

# a_ins = A(4., 3.)
# def f2(a):
#     return jnp.ones(5) + a.a - 4 * a.b

# eqx.filter_jacfwd(f2)(a_ins)

# Backup

In [211]:
# primals = [sas_Q, α_Q, α_ET, k1, C_eq, C_Q_0, P_Q_old]
# tangents = [sas_Q_tangents, 1.0, 0.0, jnp.ones(k1.shape), jnp.ones(C_eq.shape), 
#             jnp.ones(C_Q_0.shape), jnp.ones(P_Q_old.shape)]


In [212]:
# sas_Q_tangents = jtu.tree_map(lambda _: 0.0, sas_Q)
# sas_Q_tangents = eqx.tree_at(
#     lambda t: tuple(getattr(t, p) for p in ["a","b"]),
#     sas_Q,
#     replace=tuple(1.0 for _ in ["a","b"]),
# )

In [ ]:
# r, δr = eqx.filter_jvp(evolve, primals, tangents)


In [ ]:
def solve_ttd_sTmT_aug(
    aug_J_full, aug_C_J, aug_Q, aug_ET, dt,
    aug_sTmT_0, aug_ST_top_0, aug_ST_bot_0, aug_sTmT_init,  
    aug_sas_Q, aug_sas_ET,
    aug_α_Q, aug_α_ET, aug_k1, aug_C_eq, 
    *, 
    solver, f
):
    jax.debug.print("aug_Q: {x}", x=aug_Q)
    # aug_J_full: (nτ, nt, 2)
    # aug_C_J: (nt, 2)
    # aug_Q: (nt, 2)
    # aug_ET: (nt, 2)
    # dt: (1,)
    # aug_sTmT_0: (nt, 1+nm, 2)
    # aug_ST_top_0: (nt+1, 2)
    # aug_ST_bot_0: (nt+1, 2)
    # aug_sTmT_init: (nτ, 1+nm, 2)
    # aug_sas_Q, aug_sas_ET: (2,)
    # aug_α_Q, aug_α_ET: (2,)
    # aug_k1, aug_C_eq: (nm, 2)

    aug_sas_funcs = [aug_sas_Q, aug_sas_ET]
    aug_other_args = [aug_α_Q, aug_α_ET, aug_k1, aug_C_eq]
    
    def step_τ(states, x):
        # aug_sTmT_init_τ : (1+nm, 2)
        # aug_J_τ: (nt, 2)
        aug_sTmT_init_τ, aug_J_τ = x
        aug_sT_init_τ = aug_sTmT_init_τ[0,:]  # (2,)
    
        # aug_sTmT : (nt, 1+nm, 2)
        # aug_ST_top, ST_bot : (nt+1, 2)
        aug_sTmT, aug_ST_top, aug_ST_bot = states
        
        # Get fluxes and sas arguments
        aug_fluxes = [aug_J_τ, aug_C_J, aug_Q, aug_ET]

        # TODO: this should be the function input!
        aug_sas_args_Q = [aug_Q[:,[0]], aug_Q[:,[1]]]
        aug_sas_args_ET = [aug_ET[:,[0]], aug_ET[:,[1]]]
        aug_sas_args = [aug_sas_args_Q, aug_sas_args_ET]
    
        # Solve the ODE system
        # aug_sTmT_new: (nt, 1+nm, 2)
        # aug_pQET_new: (nt, 2, 2)
        # aug_mQET_new: (nt, nm, 2, 2)
        # aug_mRET_new: (nt, nm, 2)
        aug_sTmT_new, aug_pQET_new, aug_mQET_new, aug_mRET_new = solver(
            aug_sTmT, aug_ST_top[:-1,:], aug_ST_bot[1:,:], dt, 
            *aug_fluxes, *aug_sas_funcs, *aug_sas_args, *aug_other_args,
            f=f
        )
        aug_sT_new = aug_sTmT_new[:,0,:]  # (nt, 2)

        # Update ST
        aug_ST_top_new = aug_ST_bot  # (nt+1, 2)
        aug_ST_bot_new = jnp.concat(
            [aug_sT_init_τ[None,:] * dt, aug_ST_bot[1:,:] + aug_sT_new * dt], axis=0
        )  # (nt+1, 2)
        
        # Variables as the initial condition to the next step
        aug_sTmT_new_rotate = jnp.concat(
            [aug_sTmT_init_τ[None,...], aug_sTmT_new[:-1,:,:]], axis=0
        )  # (nt, 1+nm, 2)

        # Variables to save
        state_to_save = [aug_sTmT_new, aug_mQET_new, aug_pQET_new]
        
        return (aug_sTmT_new_rotate, aug_ST_top_new, aug_ST_bot_new), state_to_save
    
    _, state_to_save = jax.lax.scan(
        step_τ, (aug_sTmT_0, aug_ST_top_0, aug_ST_bot_0), (aug_sTmT_init, aug_J_full)
    )

    aug_sTmTs, aug_mQETs, aug_pQETs = state_to_save
    jax.debug.print("sTmTs: {x}", x=aug_sTmTs[...,0])
    jax.debug.print("tan_sTmTs: {x}", x=aug_sTmTs[...,1])

    return aug_sTmTs, aug_mQETs, aug_pQETs


In [ ]:
# solve_ode_ift = jax.custom_jvp(solve_ode, nondiff_argnums=(0,))
solve_ttd_sTmT_ift = eqx.filter_custom_jvp(solve_ttd_sTmT)

@solve_ttd_sTmT_ift.def_jvp
def solve_ttd_sTmT_ift_jvp(primals, tangents, *, solver, f):
    J_full, C_J, Q, ET, dt,\
    sTmT_0, ST_top_0, ST_bot_0, sTmT_init,\
    sas_Q, sas_ET,\
    α_Q, α_ET, k1, C_eq = primals
    
    δJ_full, δC_J, δQ, δET, _,\
    δsTmT_0, δST_top_0, δST_bot_0, δsTmT_init,\
    δsas_Q, δsas_ET,\
    δα_Q, δα_ET, δk1, δC_eq = tangents

    print(C_eq, sas_Q)
    print(tangents)

    # TODO: 
    def f_aug(
        aug_sTmT, aug_ST, dt, 
        aug_J_τ, aug_C_J_τ,
        aug_Q, aug_ET, 
        aug_sas_Q, aug_sas_ET,
        aug_sas_arg_Q, aug_sas_arg_ET,
        aug_α_Q, aug_α_ET, aug_k1, aug_C_eq
    ):
        primals = [
            aug_sTmT[...,0], aug_ST[...,0], dt, 
            aug_J_τ[...,0], aug_C_J_τ[...,0],
            aug_Q[...,0], aug_ET[...,0], 
            aug_sas_Q[0], aug_sas_ET[0],
            aug_sas_arg_Q[0], aug_sas_arg_ET[0],
            aug_α_Q[0], aug_α_ET[0], aug_k1[...,0], aug_C_eq[...,0]
        ]
        tangents = [
            aug_sTmT[...,1], aug_ST[...,1], dt, 
            aug_J_τ[...,1], aug_C_J_τ[...,1],
            aug_Q[...,1], aug_ET[...,1],
            aug_sas_Q[1], aug_sas_ET[1],
            aug_sas_arg_Q[0], aug_sas_arg_ET[0],  # TODO!!
            aug_α_Q[1], aug_α_ET[1], aug_k1[...,1], aug_C_eq[...,1]
        ]
        primal_dot, tangent_dot = eqx.filter_jvp(f, primals, tangents)
        states_dot, pQET_dot, mQET_dot, mR_dot = primal_dot
        δstates_dot, δpQET_dot, δmQET_dot, δmR_dot = tangent_dot
        return jnp.stack([states_dot, δstates_dot], axis=-1), jnp.stack([pQET_dot, δpQET_dot], axis=-1), \
               jnp.stack([mQET_dot, δmQET_dot], axis=-1), jnp.stack([mR_dot, δmR_dot], axis=-1)
    
    aug_states = solve_ttd_sTmT_aug(
        jnp.stack([J_full, δJ_full], axis=-1), jnp.stack([C_J, δC_J], axis=-1), 
        jnp.stack([Q, δQ], axis=-1), jnp.stack([ET, δET], axis=-1), dt,
        jnp.stack([sTmT_0, δsTmT_0], axis=-1), jnp.stack([ST_top_0, δST_top_0], axis=-1),
        jnp.stack([ST_bot_0, ST_bot_0], axis=-1), jnp.stack([sTmT_init, δsTmT_init], axis=-1),
        [sas_Q, δsas_Q], [sas_ET, δsas_ET],
        jnp.stack([α_Q, δα_Q], axis=-1), jnp.stack([α_ET, δα_ET], axis=-1),
        jnp.stack([k1, δk1], axis=-1), jnp.stack([C_eq, δC_eq], axis=-1),
        f=f_aug, solver=solver
    )
    aug_sTmTs, aug_mQETs, aug_pQETs = aug_states

    # TODO: Check the shape
    sTmTs, sTmTs_dot = aug_sTmTs[...,0], aug_sTmTs[...,1]
    mQETs, mQETs_dot = aug_mQETs[...,0], aug_mQETs[...,1]
    pQETs, pQETs_dot = aug_pQETs[...,0], aug_pQETs[...,1]
    # print(aug_pQETs)
    return (sTmTs, mQETs, pQETs), (sTmTs_dot, mQETs_dot, pQETs_dot)
